# 10 · Logistic regression for incident NEPC: risk score vs Gleason vs TP53/PTEN/RB1

Which pre-landmark information separates patients who go on to NEPC? Each predictor set is fit as its own
unpenalized logistic regression of **incident NEPC after ADT + landmark**, per subcohort × landmark. All logic lives in
`COMPASS/multivariable_models/nepc_subtyping_lr.py`.

| Model | Predictors | Odds ratio per |
|---|---|---|
| `risk` | frozen federated (no-DFCI) **platinum-trained XGBoost** score, as notebook 04 scored it | SD (z-scored within each fit) |
| `gleason` | Gleason score closest to ADT + landmark among those available before it | point |
| `trio` | `TP53_ANY` + `PTEN_ANY` + `RB1_ANY` | altered vs wild-type |
| `tp53` / `pten` / `rb1` | one gene each | altered vs wild-type |

**Outcome.** `event` from the `endpoint == "nepc"` rows of 04's `transfer_scores.csv` (XGBoost, `config == "both"`):
1 = first NEPC after ADT + landmark; patients with NEPC or censoring at or before the landmark are not at risk and
are not in the fit. 0 = everyone else at risk, whatever their follow-up, so this is a binary
"ever NEPC during follow-up" label and not a fixed-horizon one.

**TSG timing.** At each landmark, use the report closest to ADT + landmark among reports available
strictly before it, for both cases and controls. Same-date reports are combined by gene-wise maximum;
a sequenced patient's uncalled gene counts as wild-type. Patients without a pre-landmark report are
excluded from somatic models and the matched population. `n_sequenced` counts patients with a report
available before that landmark.

**Populations.** `available`: each model on every patient with its own predictor. `matched`: every model on the
patients with risk, Gleason and an eligible specimen, with paired-bootstrap ΔAUC vs `risk`. AUC is out-of-fold
(stratified 5-fold) with a patient-bootstrap 95% CI; ORs, Wald CIs and LRT p-values come from the full-population fit.
Fewer than `MIN_EVENTS` events or non-events → count-only row. Any failed CV fold suppresses that
model’s AUC, CI and paired ΔAUC (`n_failed_folds`); `converged` describes the full-population fit.

**Prerequisites.** Notebook 04 has written `transfer_scores.csv`; 01 has written the metastatic MRN lists.
`metastatic_adt` / `metastatic_llm` labels are retrospective strata (they use follow-up), not landmark-time predictors.

In [ ]:
import os, sys
sys.path.insert(0, "../../pipeline")
import compass_pipeline as cp
for _p in (str(cp.PROJECT_ROOT),):
    if _p not in sys.path: sys.path.insert(0, _p)
from pathlib import Path
import pandas as pd
from IPython.display import display
from survival_common import federated_endpoint_transfer as fet
from COMPASS.multivariable_models import nepc_subtyping_lr as nsl
fcc = nsl.fcc

DATA_ROOT  = Path(cp._PROFILE_OUTPUT_ROOT)
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(DATA_ROOT / "rhino_data_v2")))

# >>> Edit <<<
COHORTS     = ("all", "metastatic_adt", "metastatic_llm")   # "all" = every patient 04 scored
LANDMARKS   = (0, 90, 180)
MIN_EVENTS  = 5
N_FOLDS     = 5
N_BOOTSTRAP = 200
SEED        = 0

SCORES_PATH           = RHINO_ROOT / "federated_transfer_adt_no_DFCI" / "all_models" / "transfer_scores.csv"
DATA_PATH             = DATA_ROOT / "longitudinal_prediction_data_adt.csv"
GLEASON_PATH          = fcc.sg.DEFAULT_GLEASON_PATH
SOMATIC_PATH          = fcc.sg.DEFAULT_SOMATIC_PATH
SOMATIC_MANIFEST_PATH = fcc.sg.DEFAULT_SOMATIC_MANIFEST_PATH
OUT_DIR               = DATA_ROOT / "nepc_subtyping_lr"   # protected data root; aggregate tables only

## Load inputs

In [ ]:
scores = nsl.load_transfer_scores(SCORES_PATH, family="xgboost_cox", config="both", endpoint="nepc")
scores = scores[scores["landmark_days"].isin(LANDMARKS)]
local = fet.read_local_frame(DATA_PATH, ("nepc",))
clin_gleason, clin_somatic, clin_somatic_features = fcc.load_clinical_sources(
    GLEASON_PATH, SOMATIC_PATH, SOMATIC_MANIFEST_PATH)
somatic_features = fcc.adt_anchored_somatic(
    local, LANDMARKS, somatic=clin_somatic, somatic_features=clin_somatic_features)
# ADT clock, no specimen: Gleason available before ADT + landmark (the somatic columns stay empty here).
gleason_features = fcc.build_clinical_features(local, LANDMARKS, gleason=clin_gleason)
members = nsl.cohort_members({c: None if c == "all" else cp.cohort_mrn_list_path(c) for c in COHORTS})

print(f"{scores[nsl.ID_COL].nunique():,} scored patients; landmarks {sorted(scores['landmark_days'].unique())}")
print(f"{somatic_features[nsl.ID_COL].nunique():,} patients with a report before at least one landmark")
for c, ids in members.items():
    print(f"  {c}: {'no restriction' if ids is None else f'{len(ids):,} MRNs'}")

## LR frame and attrition

In [ ]:
frame = nsl.build_lr_frame(scores, gleason_features, somatic_features)
models, coefficients, counts = nsl.run_lr(
    frame, members, LANDMARKS, min_events=MIN_EVENTS, n_folds=N_FOLDS,
    n_bootstrap=N_BOOTSTRAP, seed=SEED)
display(counts.set_index(["cohort", "landmark_days"]))

## Write outputs

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
for name, df in (("lr_models", models), ("lr_coefficients", coefficients), ("lr_counts", counts)):
    df.to_csv(OUT_DIR / f"{name}.csv", index=False)
print(f"Wrote lr_models.csv, lr_coefficients.csv, lr_counts.csv to {OUT_DIR}")

## Results

Odds ratios (95% CI) and Wald p per term, then out-of-fold AUC (95% CI) per model, and ΔAUC vs `risk` on the matched patients.

In [ ]:
def fmt(est, lo, hi, digits=2):
    return pd.Series([f"{e:.{digits}f} ({l:.{digits}f}–{h:.{digits}f})" if pd.notna(e) else ""
                      for e, l, h in zip(est, lo, hi)], index=est.index)

idx = ["cohort", "landmark_days"]
order = {m: i for i, m in enumerate(nsl.MODEL_SPECS)}
for population in nsl.POPULATIONS:
    co = coefficients[coefficients["population"].eq(population)].assign(
        OR=lambda d: fmt(d["odds_ratio"], d["or_lo"], d["or_hi"]))
    print(f"\n=== {population}: odds ratio (95% CI) ===")
    terms = sorted(dict.fromkeys(zip(co["model"], co["term"])), key=lambda k: order[k[0]])  # fit order within model
    display(co.pivot_table(index=idx, columns=["model", "term"], values="OR", aggfunc="first").reindex(columns=terms))
    print(f"=== {population}: Wald p ===")
    display(co.pivot_table(index=idx, columns=["model", "term"], values="p_wald").reindex(columns=terms).apply(lambda col: col.map(lambda p: f"{p:.2g}" if pd.notna(p) else "")))

    mo = models[models["population"].eq(population)].assign(
        AUC=lambda d: fmt(d["auc"], d["auc_lo"], d["auc_hi"], 3))
    cols = sorted(mo["model"].unique(), key=order.get)
    print(f"=== {population}: out-of-fold AUC (95% CI) ===")
    display(mo.pivot_table(index=idx, columns="model", values="AUC", aggfunc="first").reindex(columns=cols))
    display(mo.pivot_table(index=idx, columns="model", values=["n", "events"], aggfunc="first"))
    if population == "matched":
        mo = mo.assign(dAUC=lambda d: fmt(d["delta_auc_vs_risk"], d["delta_auc_lo"], d["delta_auc_hi"], 3))
        print("=== matched: ΔAUC vs risk (95% CI) ===")
        display(mo[mo["model"].ne(nsl.REFERENCE_MODEL)]
                  .pivot_table(index=idx, columns="model", values="dAUC", aggfunc="first")
                  .reindex(columns=[c for c in cols if c != nsl.REFERENCE_MODEL]))

notes = models[models["note"].fillna("").ne("")]
if len(notes):
    print("\nModels with notes (not fit, constant predictor, separation, or CV failure):")
    display(notes[["cohort", "landmark_days", "population", "model", "n", "events", "n_failed_folds", "note"]])